# raglib — демо всего функционала

Поэтапный проход по возможностям библиотеки: распознавание → индекс → поиск
(`bm25`/`vector`/`hybrid`/`grep`) → навигация → работа с **разделами без номера** →
агентский поиск с реальным LLM (**OpenRouter**, как в deep_agent).

Данные: реальные уставы из `work/cache/*/text.md` (если есть локально) + синтетический
регламент для разделов без номера; иначе — синтетический устав. Эмбеддинги офлайновые
(`HashingEmbeddings`).

> ⚠️ **Конфиденциальность и стоимость.** Реальные уставы загружаются по пути (каталог
> `work/` в `.gitignore`, репозиторий публичный) — не коммитьте ноутбук с выполненным
> выводом (*Kernel → Restart & Clear Output*). Ячейка агентского поиска с подключённым
> LLM **отправляет фрагменты документов в OpenRouter** и расходует токены.

## Настройка ядра

Запускайте на ядре **«Python (raglib)»** — это venv, где установлен raglib
(зарегистрировать: `python -m ipykernel install --user --name raglib
--display-name "Python (raglib)"`).

Ячейка ниже проверяет окружение. Свежие `langchain`/`openai` требуют
`typing_extensions>=4.13` (PEP 728, `extra_items`); в старом ядре импорт падает с
`_TypedDictMeta.__new__() got an unexpected keyword argument 'extra_items'`. Если
версия старая — ячейка её обновит; после этого **перезапустите ядро** и запустите
ноутбук заново.

In [ ]:
import sys, subprocess, importlib.util
from importlib.metadata import version

print("интерпретатор ядра:", sys.executable)

def _major_minor(pkg):
    try:
        return tuple(int(x) for x in version(pkg).split(".")[:2])
    except Exception:
        return None

te = _major_minor("typing_extensions")
if te is None or te < (4, 13):
    print(f"typing_extensions={te}: обновляю (нужно >=4.13 для langchain/openai)…")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "typing_extensions>=4.13"], check=False)
    print("⚠️  ПЕРЕЗАПУСТИТЕ ЯДРО (Kernel → Restart) и запустите ноутбук заново.")
else:
    print("typing_extensions OK:", version("typing_extensions"))

if importlib.util.find_spec("raglib") is None:
    print("⚠️  raglib не найден в этом ядре — выберите ядро «Python (raglib)» "
          "или выполните `pip install -e .` в папке raglib.")

In [ ]:
import json, os, re, shutil, subprocess, tempfile
from pathlib import Path

from raglib import RagIndex
from raglib.embeddings import HashingEmbeddings

def find_repo_root() -> Path:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / ".git").exists() or (base / "work" / "cache").exists():
            return base
    return Path.cwd()

def show(hits, n=95):
    for h in hits:
        tag = f" [{h.verdict}]" if getattr(h, "verdict", None) else ""
        print(f"  [{h.locator}]{tag}  ({h.method}, score={h.score:.3f})")
        print(f"      {' '.join(h.text.split())[:n]}…")

REPO = find_repo_root()
print("repo root:", REPO)

## Шаг 1 — Распознавание: файл → markdown

raglib **не** распознаёт PDF сам: это делает целевая система (в deep_agent —
docling), а на вход подаётся markdown. Ниже собираем корпус `.md`. Хотите свой
PDF — укажите `PDF_PATH`: если в системе есть `docling`, ячейка распознает его
(теми же флагами, что deep_agent: `--no-ocr --no-tables --image-export-mode
placeholder`) и добавит в корпус.

In [ ]:
workdir = Path(tempfile.mkdtemp(prefix="raglib_demo_"))

# синтетика: fallback-устав (с оглавлением) + регламент (разделы без номера)
УСТАВ_SYNTH = """# УСТАВ ООО «Ромашка»

Утверждён протоколом Общего собрания участников № 01-2025.

## СОДЕРЖАНИЕ

- Статья 12. Наблюдательный совет
- Статья 13. Крупные сделки

## Статья 12. НАБЛЮДАТЕЛЬНЫЙ СОВЕТ

12.1. К компетенции Наблюдательного совета относится предварительное одобрение сделок Общества.

## Статья 13. КРУПНЫЕ СДЕЛКИ

13.1. Крупной сделкой признаётся сделка, стоимость которой превышает 25 процентов балансовой стоимости активов Общества.

13.2. Решение о согласии на совершение крупной сделки принимается Общим собранием участников Общества.
"""
РЕГЛАМЕНТ = """# Регламент доступа к информационным системам

Настоящий документ действует с даты его утверждения.

## Назначение

Регламент определяет порядок предоставления доступа сотрудников к информационным системам банка.

## Отзыв доступа

Доступ отзывается автоматически в день увольнения сотрудника либо по требованию службы информационной безопасности.
"""

real = sorted((REPO / "work" / "cache").glob("*/text.md"))
if real:
    for i, src in enumerate(real, 1):
        shutil.copyfile(src, workdir / f"устав_{i}.md")   # обезличенные имена
    print(f"реальные уставы: {len(real)}")
else:
    (workdir / "устав.md").write_text(УСТАВ_SYNTH, encoding="utf-8")
    print("реальные уставы не найдены — синтетический пример")
(workdir / "регламент.md").write_text(РЕГЛАМЕНТ, encoding="utf-8")

# --- опционально: распознать свой PDF через docling -------------------------
PDF_PATH = None   # ← например "/Users/.../Устав.pdf"
def _find_docling():
    exe = shutil.which("docling")
    if exe:
        return exe
    for c in ("/opt/miniconda3/bin/docling", str(Path.home() / "miniconda3/bin/docling")):
        if Path(c).exists():
            return c
    return None
if PDF_PATH:
    dl = _find_docling()
    if dl:
        subprocess.run([dl, PDF_PATH, "--to", "md", "--no-ocr", "--no-tables",
                        "--image-export-mode", "placeholder", "--output", str(workdir)],
                       check=True)
        print("распознан PDF:", Path(PDF_PATH).name)
    else:
        print("docling не найден — пропускаю распознавание PDF")

print("корпус:", *sorted(p.name for p in workdir.glob("*.md")))

## Шаг 2 — Сборка индекса

`RagIndex.build` распознаёт (здесь вход уже `.md`), режет на **целые пункты**,
строит BM25 (в памяти на загрузке) и FAISS (chunks + sections). `embeddings=None`
дал бы BM25-only индекс.

In [ ]:
index = RagIndex.build(
    inputs=workdir,
    index_dir=workdir / "index",
    embeddings=HashingEmbeddings(),        # прод: langchain_gigachat.GigaChatEmbeddings(...)
    bm25_normalizer="auto",                # сам берёт stem/lemma/none по доступности
)
print("документы:", index.documents)
print("счётчики :", index.manifest["counts"])
print("BM25     :", index.manifest["bm25"]["normalizer"],
      "| эмбеддинги:", index.manifest["embeddings"]["model"])

charter_docs = [d for d in index.documents if d != "регламент"]
primary = charter_docs[0]           # основной устав для примеров
print("основной документ:", primary)

## Шаг 3 — Структура: оглавление

`toc()` — читаемый очерк (ключ + заголовок). `preview=True` добавляет первую
содержательную фразу раздела, `clauses=True` — номера пунктов раздела.
`toc_entries()` — то же структурно (`list[TocEntry]`).

In [ ]:
print(index.toc(doc=primary, preview=True))
print("\n--- первые 5 структурных строк (toc_entries) ---")
for e in index.toc_entries(doc=primary)[:5]:
    print(f"  key={e.key!r:8} level={e.level}  {e.title[:60]}")

## Шаг 4 — Поиск: bm25 / vector / hybrid

Единый контракт: результат — **целый пункт с номером**. `bm25` — лексика,
`vector` — смысл (FAISS-косинус), `hybrid` — их объединение через RRF.

In [ ]:
query = "компетенция общего собрания" if any("собрани" in e.title.lower()
        for e in index.toc_entries(doc=primary)) else "одобрение крупных сделок"
print("запрос:", query, "\n")
for method in ("bm25", "vector", "hybrid"):
    print(f"=== {method} ===")
    show(index.search(query, method=method, top_k=2, doc=primary))

## Шаг 5 — Фильтры и стратегия `tree`

`doc=` / `section=` сужают область; `strategy="tree"` — coarse-to-fine
(сначала разделы по смыслу, затем пункты внутри).

In [ ]:
sec = next((e.key for e in index.toc_entries(doc=primary) if e.key[:1].isdigit()), None)
print(f"фильтр section={sec!r} (doc={primary!r}):")
show(index.search("общество", method="bm25", top_k=2, doc=primary, section=sec))
print("\nstrategy='tree' (разделы → пункты):")
show(index.search(query, method="vector", strategy="tree", top_k=2, doc=primary))

## Шаг 6 — Точный поиск: `grep`

Регулярное выражение по пунктам; выдача — те же целые пункты (score = число
совпадений). Полезно для номеров, процентов, ссылок на статьи.

In [ ]:
show(index.grep(r"\d+\s*процент", top_k=3, doc=primary))

## Шаг 7 — Навигация: find_section / read_section

`find_section` находит раздел по ключу, заголовку, превью содержимого или по
смыслу (`semantic=True`, нужен вектор-индекс). `read_section` отдаёт раздел
целиком, с подразделами.

In [ ]:
ref = index.find_section("общее собрание" if primary != "регламент" else "доступ",
                         top_k=1)
print("find_section →", ref[0] if ref else "—")
try:
    sref = index.find_section("порядок принятия решений", semantic=True, top_k=1)
    print("find_section(semantic=True) →", sref[0] if sref else "—")
except Exception as e:
    print("semantic пропущен:", e)

if ref:
    body = index.read_section(ref[0].doc_id, ref[0].key)
    print(f"\nread_section({ref[0].doc_id!r}, {ref[0].key!r}) — {len(body)} симв.:")
    print("  " + " ".join(body.split())[:200] + "…")

## Шаг 8 — Разделы и пункты БЕЗ номера

Раздел без номера (преамбула, «Назначение», в реальном уставе — «Увеличение
уставного капитала», `## УТВЕРЖДЕНО`) получает **адресуемый ключ `§N`**. У пункта
без номера `clause_number` пуст, но `breadcrumb` даёт раздел-владелец, а
`locator` — всегда непустую ссылку.

In [ ]:
syn = [(d, e) for d in index.documents
       for e in index.toc_entries(doc=d) if e.key.startswith("§")]
print("разделы без номера (§-ключ):")
for d, e in syn[:10]:
    print(f"  {d}: [{e.key}] {e.title[:55]}")

print("\nпоиск в регламенте (всё без номера) — есть провенанс и locator:")
for h in index.search("отзыв доступа увольнение", method="bm25", doc="регламент", top_k=2):
    print(f"  clause_number={h.clause_number!r:4}  locator={h.locator!r:34}  breadcrumb={h.breadcrumb!r}")

print("\nread_section по §-ключу == по заголовку:",
      index.read_section("регламент", "§3") == index.read_section("регламент", "Отзыв доступа"))

## Шаг 9 — Оглавление не засоряет поиск

`## СОДЕРЖАНИЕ` перечисляет все статьи и матчился бы почти на любой запрос, приходя
с пустым номером. raglib исключает его из поиска, но оставляет в навигации.

In [ ]:
def has_toc(doc):
    return any(re.match(r"(?i)(содержание|оглавление)$", e.title.strip())
              for e in index.toc_entries(doc=doc))

toc_docs = [d for d in index.documents if has_toc(d)]
if toc_docs:
    d = toc_docs[0]
    hits = index.search("общее собрание совет директоров сделки", method="bm25", doc=d, top_k=8)
    in_search = any(h.text.lstrip().lower().startswith(("## содержание", "## оглавление"))
                    for h in hits)
    print(f"документ {d!r}: оглавление в выдаче поиска? {in_search}")
    print(f"                оглавление в toc()?        {has_toc(d)}")
else:
    print("в корпусе нет документа с оглавлением — пропускаем")

## Шаг 10 — Анатомия `SearchHit`

Всё, что несёт один результат: номер, стабильный `locator`, путь-провенанс,
документ, метод, оценка.

In [ ]:
h = index.search(query, method="hybrid", top_k=1, doc=primary)[0]
for f in ("clause_number", "locator", "doc_name", "method", "score",
          "section_path", "section_titles", "breadcrumb"):
    print(f"  {f:14} = {getattr(h, f)!r}")
print(f"  text (первые 80) = {' '.join(h.text.split())[:80]!r}…")

## Шаг 11 — Подключение LLM (OpenRouter, как в deep_agent)

Читаем `.env` (ключ `OPENROUTER_API_KEY`, модель `MODEL_NAME`) и собираем
`langchain_openai.ChatOpenAI` теми же параметрами, что `main._build_model` в
deep_agent (`temperature=0.1`, `max_tokens=8192`). После рефактора raglib берёт
LangChain-модель **напрямую**. Нет ключа → `llm=None`, и агентский поиск
деградирует в hybrid.

`.env` ищется **вверх по дереву от текущей папки** (как `python-dotenv`), либо
задайте путь вручную в `ENV_PATH`.

In [ ]:
ENV_PATH = None   # ← при необходимости укажите путь к .env явно, например "/Users/.../.env"

def load_env(env_path=None):
    """Найти .env (вверх от cwd) и загрузить в os.environ. Возвращает путь и
    число ключей. Пустые/отсутствующие переменные заполняет, явно заданные
    непустые — не трогает. Использует python-dotenv, если он есть."""
    p = Path(env_path) if env_path else None
    if p is None:
        for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
            if (base / ".env").is_file():
                p = base / ".env"
                break
    if not p or not p.is_file():
        return None, 0
    values = {}
    try:
        from dotenv import dotenv_values                     # как в deep_agent
        values = {k: v for k, v in dotenv_values(p).items() if v is not None}
    except ImportError:                                       # надёжный ручной парсер
        for raw in p.read_text(encoding="utf-8").splitlines():
            s = raw.strip()
            if not s or s.startswith("#") or "=" not in s:
                continue
            if s.startswith("export "):
                s = s[len("export "):]
            k, v = s.split("=", 1)
            k, v = k.strip(), v.strip()
            if len(v) >= 2 and v[0] == v[-1] and v[0] in ('"', "'"):   # снять кавычки
                v = v[1:-1]
            values[k] = v
    for k, v in values.items():
        if not os.environ.get(k):        # не затираем явно заданное непустое значение
            os.environ[k] = v
    return p, len(values)

llm = None
try:
    from langchain_openai import ChatOpenAI
    envf, n = load_env(ENV_PATH)
    print(f".env: {envf}  (ключей: {n})" if envf else ".env не найден")
    if os.getenv("OPENROUTER_API_KEY"):
        llm = ChatOpenAI(
            model=os.getenv("MODEL_NAME", "deepseek/deepseek-v4-flash"),
            openai_api_base=os.getenv("OPENROUTER_BASE_URL", "https://openrouter.ai/api/v1"),
            openai_api_key=os.getenv("OPENROUTER_API_KEY"),
            temperature=float(os.getenv("LLM_TEMPERATURE", "0.1")),
            max_tokens=int(os.getenv("LLM_MAX_OUTPUT_TOKENS", "8192")),
            max_retries=int(os.getenv("LLM_MAX_RETRIES", "2")),
        )
        print("LLM подключён:", llm.model_name, "@ OpenRouter")
    else:
        print("OPENROUTER_API_KEY не найден → llm=None (агент пойдёт как hybrid)."
              " Задайте ENV_PATH или экспортируйте переменную.")
except Exception as e:
    print("LLM не подключён:", type(e).__name__, e, "→ llm=None")
    if "extra_items" in str(e):
        print("  → устаревшая typing_extensions в ядре: запустите ячейку «Настройка"
              " ядра», перезапустите ядро; либо выберите ядро «Python (raglib)».")

## Шаг 12 — Агентский поиск

Код-управляемый цикл: **PLAN** (LLM: мульти-запросы + аспекты) → **RETRIEVE** (код:
hybrid + эвристики) → **REFLECT** (LLM: вердикты relevant/partial/irrelevant) →
**REFINE**. Возвращает `AgenticResult` (`.hits`, `.trace`, `.degraded`,
`.iterations`, `.llm_calls`).

Важно не путать два режима:
* `llm=None` → **штатный** чистый hybrid без рефлексии, `degraded=False`;
* `degraded=True` → **деградация**: LLM подключён, но сбоит (сеть/парсинг), и агент
  безопасно откатывается в hybrid — «никогда не хуже hybrid».

Ниже: сначала базовый режим без LLM, затем — полный цикл с подключённым LLM.

In [ ]:
prompt = ("Кто одобряет крупные сделки и с какими порогами?" if primary != "регламент"
          else "Как отзывается доступ сотрудника?")
print("вопрос:", prompt)

print("\n=== без LLM (llm=None → чистый hybrid, это НЕ деградация) ===")
base = index.agentic_search(prompt, llm=None, top_k=5)
print(f"degraded={base.degraded} iterations={base.iterations} llm_calls={base.llm_calls}")
show(base.hits)

if llm is None:
    print("\nLLM не подключён (см. Шаг 11) — агентский цикл не показать. "
          "Подключите LLM и перезапустите ячейку.")
else:
    print("\n=== с LLM (полный цикл PLAN → RETRIEVE → REFLECT → REFINE) ===")
    res = index.agentic_search(prompt, llm=llm, top_k=5)
    print(f"degraded={res.degraded} iterations={res.iterations} llm_calls={res.llm_calls}")
    for step in res.trace:                       # трасса каждого шага
        st = step.get("step")
        if st == "plan" and "queries" in step:
            print("  PLAN queries:", step["queries"])
            print("  PLAN aspects:", step["aspects"])
        elif st == "refine" and "queries" in step:
            print("  REFINE queries:", step["queries"])
        elif st == "graded":
            print("  REFLECT:", step["verdicts"])
        elif st == "degrade":
            print("  DEGRADE:", step.get("reason"))
    print("  --- пункты (в скобках вердикт LLM) ---")
    show(res.hits)

In [ ]:
res.hits

## Шаг 13 — Жизненный цикл индекса

Индекс персистентный: `RagIndex.load` открывает готовый (без пересборки),
`bm25_normalizer=` можно переопределить на загрузке. `delete_index` — валидирует
и удаляет папку (идемпотентно).

In [ ]:
reopened = RagIndex.load(workdir / "index", embeddings=HashingEmbeddings())
print("переоткрыт, документов:", len(reopened.documents))
print("на диске лежит:", *(p.name for p in (workdir / "index").iterdir()))
print("удаление индекса:", RagIndex.delete_index(workdir / "index"))
print("повторное удаление (идемпотентно):", RagIndex.delete_index(workdir / "index"))

---
Это весь публичный контур raglib: `build/load/search/grep/toc/toc_entries/
find_section/read_section/agentic_search/delete`. В контуре замените
`HashingEmbeddings` → `GigaChatEmbeddings(...)` и OpenRouter-`ChatOpenAI` →
`langchain_gigachat.GigaChat(...)` — код не меняется.

**Перед коммитом при работе на реальных данных:** очистите вывод
(*Kernel → Restart & Clear Output*).